# Account-level ELT → account-level PLT

Simulation engine. Companion to `elt_to_ylt`, which works at portfolio grain.

## Why the portfolio algorithm cannot be reused

| | portfolio grain | account grain |
|---|---|---|
| one ELT row is | one event | one `(event, account)` pair |
| frequency | Poisson superposition over rows, then sample which row | Poisson per **event**, then loop accounts within |
| severity | `rng.beta(a, b)` per row, independent | `beta.ppf(norm.cdf(shared + private))` |
| `sd_corr` | folded into `sd_tot`, treated as independent | drives a shock **shared** across the footprint |

Feeding an account ELT to the portfolio algorithm fails *silently*. Pooling `rate` across
`(event, account)` rows inflates frequency by roughly the accounts-per-event count, and sampling one
row per occurrence makes accounts independent — so one hurricane hitting eight accounts becomes eight
separate occurrences hitting one account each. The portfolio tail collapses and every diversification
and marginal-impact number is wrong, with no error raised.

## Severity convention

    sd_tot = sd_indep + sd_corr                       (linear, RMS-style)
    damage ratio ~ Beta(mean = mean_loss/exposure, sd = sd_tot/exposure)
    loss = damage ratio × exposure

Correlation comes from a Gaussian copula:

    u    = Φ( w_corr · z_shared + w_indep · z_private )
    loss = BetaPPF(u; a, b) × exposure

with `w_corr, w_indep ∝ sd_corr, sd_indep`, normalised so `w_corr² + w_indep² = 1` to keep the mixed
variable standard normal. The Beta **marginal** is identical to `rng.beta(a, b)` — a quantile
transform of a uniform *is* a Beta. Only the dependence changes.

## Required ELT columns

`event_id, accnt_no, rate, mean_loss, sd_indep, sd_corr, exposure`

`rate` must be constant within `event_id` — it is a property of the event, not the account. Asserted.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats

REQUIRED = {"event_id", "accnt_no", "rate", "mean_loss", "sd_indep", "sd_corr", "exposure"}

## 1 · Severity helper

Method-of-moments Beta. A Beta on `[0, 1]` admits `s² < m(1−m)`; where the requested sd exceeds that
bound it is capped just below and flagged, rather than silently producing invalid parameters.

In [ ]:
def beta_params_from_moments(m, s):
    """Method-of-moments Beta from mean ratio `m` and sd ratio `s`, both in (0, 1).
    Returns (a, b, capped)."""
    m = np.asarray(m, dtype=float)
    s = np.asarray(s, dtype=float)
    s_max  = np.sqrt(m * (1.0 - m))
    capped = s >= s_max
    s  = np.where(capped, np.nextafter(s_max, 0.0) * (1 - 1e-9), s)
    nu = m * (1.0 - m) / s ** 2 - 1.0
    return m * nu, (1.0 - m) * nu, capped

## 2 · The engine

Four steps per event:

1. **Frequency** — `rng.poisson(rate, n_years)`, one draw per *event*.
2. **Dates** — a base day per year, `+1` per repeat occurrence mod 365, so two occurrences of the
   same event in one year get distinct dates. This keeps `(year_id, event_id, loss_date)` unique,
   which OEP needs.
3. **The common shock** — `z_shared = rng.standard_normal(n_occ)`, drawn *once per occurrence, before
   the account loop*. This single line is the whole mechanism.
4. **Each account in the footprint** mixes that shared shock with its own private draw.

The ELT is sorted before the loop because the RNG is consumed in that order — a stable order is what
makes the output reproducible under a seed.

In [ ]:
def elt_to_account_plt(elt, n_years, seed=42, secondary=True, days_in_year=365,
                       drop_zero=True, zero_tol=0.0, verbose=True):
    """Simulate an account-level PLT from an account-level ELT.

    Parameters
    ----------
    elt          : DataFrame with the REQUIRED columns, one row per (event_id, accnt_no)
    n_years      : number of trial years
    seed         : root seed; the same seed reproduces the table bit for bit
    secondary    : if False, use the deterministic mean_loss (accounts still co-occur)
    days_in_year : calendar length used to place occurrences on dates
    drop_zero    : drop rows with loss <= zero_tol, mirroring a vendor extract that does
                   not write sub-threshold losses; set zero_tol to your real threshold
    verbose      : print the Beta-capping warning if any occurrence hits the bound

    Returns
    -------
    DataFrame: accnt_no, event_id, year_id (1-based), loss_date, loss
    """
    missing = REQUIRED - set(elt.columns)
    if missing:
        raise ValueError(f"account ELT missing columns: {sorted(missing)}")
    if n_years < 1:
        raise ValueError("n_years must be >= 1")

    # --- rate is a property of the EVENT, not of the (event, account) pair ---
    rbe = elt.groupby("event_id")["rate"].agg(["min", "max", "first"])
    if not np.allclose(rbe["min"], rbe["max"]):
        bad = rbe.index[~np.isclose(rbe["min"], rbe["max"])].tolist()
        raise ValueError(f"rate varies within event_id for events {bad[:5]}"
                         f"{' ...' if len(bad) > 5 else ''}; it must be constant across accounts")

    rng = np.random.default_rng(seed)

    # stable order == reproducible RNG consumption
    elt    = elt.sort_values(["event_id", "accnt_no"], kind="stable")
    events = rbe.index.to_numpy()
    rates  = rbe["first"].to_numpy(dtype=float)

    frames, n_capped = [], 0

    for event_id, rate in zip(events, rates):
        legs = elt[elt["event_id"] == event_id]

        # --- 1. frequency: Poisson per EVENT, not per row -------------------
        counts = rng.poisson(rate, size=n_years)
        n_occ  = int(counts.sum())
        if n_occ == 0:
            continue
        year = np.repeat(np.arange(1, n_years + 1), counts)

        # --- 2. dates distinct within (year, event) -------------------------
        start  = np.concatenate([[0], np.cumsum(counts)])
        within = np.arange(n_occ) - np.repeat(start[:-1], counts)
        base   = rng.integers(1, days_in_year + 1, size=n_years)
        day    = 1 + (base[year - 1] - 1 + within) % days_in_year

        # --- 3. THE COMMON SHOCK: one draw per occurrence, shared by every --
        #        account exposed to it. This is the whole mechanism.
        z_shared = rng.standard_normal(n_occ) if secondary else None

        # --- 4. per account in the footprint --------------------------------
        for leg in legs.itertuples(index=False):
            mu, E = float(leg.mean_loss), float(leg.exposure)

            if not secondary or E <= 0:
                loss = np.full(n_occ, mu, dtype=float)
            else:
                m = mu / E
                s = (float(leg.sd_indep) + float(leg.sd_corr)) / E
                if s <= 0 or m <= 0 or m >= 1:
                    loss = np.full(n_occ, mu, dtype=float)          # degenerate -> mean
                else:
                    a, b, capped = beta_params_from_moments(m, s)
                    n_capped += int(np.atleast_1d(capped).sum())

                    w = np.hypot(float(leg.sd_corr), float(leg.sd_indep))
                    if w <= 0:
                        w_corr, w_indep = 0.0, 1.0
                    else:
                        w_corr, w_indep = float(leg.sd_corr) / w, float(leg.sd_indep) / w

                    z    = w_corr * z_shared + w_indep * rng.standard_normal(n_occ)
                    loss = stats.beta.ppf(stats.norm.cdf(z), a, b) * E

            frames.append(pd.DataFrame({"accnt_no": leg.accnt_no, "event_id": event_id,
                                        "year_id": year, "loss_date": day, "loss": loss}))

    if not frames:
        return pd.DataFrame(columns=["accnt_no", "event_id", "year_id", "loss_date", "loss"])

    out = pd.concat(frames, ignore_index=True)
    if drop_zero:
        out = out[out["loss"] > zero_tol].reset_index(drop=True)
    if verbose and n_capped:
        print(f"[warn] sd capped at the Beta bound for {n_capped} account-events")

    return (out.sort_values(["year_id", "event_id", "loss_date", "accnt_no"])
               .reset_index(drop=True))

## 3 · Portfolio table and the AAL check

The portfolio PLT is *derived*, never read separately — deriving it keeps the two tables consistent
by construction.

The AAL check judges closure on a **z-score**, not a flat percentage. For a compound Poisson,
`Var[annual] = Σ_e rate_e · (mean_loss² + sd_tot²)`, so the standard error of the simulated AAL is
`√(Var / n_years)`. A flat 2% tolerance would fail small, noisy accounts that are in fact fine.

In [ ]:
def aggregate_to_portfolio_plt(account_plt):
    """Sum away the account dimension: one row per occurrence."""
    return (account_plt.groupby(["year_id", "event_id", "loss_date"], as_index=False)["loss"]
            .sum().sort_values(["year_id", "event_id", "loss_date"]).reset_index(drop=True))


def validate_account_plt(account_plt, elt, n_years, z_tol=4.0):
    """Per-account AAL against the ELT analytic value, in units of Monte Carlo error."""
    analytic = elt.assign(_a=elt["rate"] * elt["mean_loss"]).groupby("accnt_no")["_a"].sum()
    var = (elt.assign(_v=elt["rate"] * (elt["mean_loss"] ** 2
                                        + (elt["sd_indep"] + elt["sd_corr"]) ** 2))
              .groupby("accnt_no")["_v"].sum())
    sim = (account_plt.groupby("accnt_no")["loss"].sum()
           .reindex(analytic.index).fillna(0.0) / n_years)
    se  = np.sqrt(var / n_years)
    res = pd.DataFrame({"aal_analytic": analytic, "aal_simulated": sim,
                        "se": se, "z": (sim - analytic) / se})
    res["ok"] = res["z"].abs() < z_tol
    return res

## 4 · A demo ELT

Replace this cell with your own load. Three events, four accounts, overlapping footprints — enough
for the correlation check below to be meaningful.

If your columns are named differently (`EventId`, `AccountId`, `MeanLoss`, …), rename before calling.
`itertuples` accesses by column name, so `leg.mean_loss` would otherwise need to become
`leg.MeanLoss`.

In [ ]:
foot = {101: ["A", "B", "C"], 102: ["A", "C", "D"], 103: ["B", "D"]}
rate = {101: 0.20, 102: 0.35, 103: 0.10}
expo = {"A": 2500.0, "B": 1800.0, "C": 900.0, "D": 3000.0}
mdr  = {("A", 101): 0.08, ("B", 101): 0.05, ("C", 101): 0.12,
        ("A", 102): 0.03, ("C", 102): 0.06, ("D", 102): 0.02,
        ("B", 103): 0.15, ("D", 103): 0.09}

recs = []
for e, accts in foot.items():
    for a in accts:
        ml = mdr[(a, e)] * expo[a]
        recs.append({"event_id": e, "accnt_no": a, "rate": rate[e], "mean_loss": ml,
                     "sd_indep": 0.45 * ml, "sd_corr": 0.30 * ml, "exposure": expo[a]})
elt = pd.DataFrame(recs)
display(elt)

## 5 · Run

In [ ]:
N_YEARS = 50_000

account_plt   = elt_to_account_plt(elt, n_years=N_YEARS, seed=42)
portfolio_plt = aggregate_to_portfolio_plt(account_plt)

print(f"account PLT {len(account_plt):,} rows | portfolio PLT {len(portfolio_plt):,} occurrences "
      f"| {len(account_plt)/len(portfolio_plt):.2f} accounts per occurrence")
display(account_plt.head(8))
display(portfolio_plt.head(4))

## 6 · Checks

Run these once on a new ELT. The fourth is the one that actually matters — everything else can pass
while the engine is quietly producing independent accounts.

In [ ]:
# 1. AAL closes against the ELT (z-scores, not percentages)
display(validate_account_plt(account_plt, elt, N_YEARS).round(4))

# 2. occurrence key unique -- OEP needs this
assert not portfolio_plt.duplicated(["year_id", "event_id", "loss_date"]).any()
print("occurrence key unique: True")

# 3. same seed -> identical table
again = elt_to_account_plt(elt, n_years=N_YEARS, seed=42, verbose=False)
assert np.array_equal(account_plt["loss"].to_numpy(), again["loss"].to_numpy())
print("reproducible under the same seed: True")
del again

In [ ]:
# 4. accounts MUST co-move within an event -- this is what sd_corr buys
ev  = portfolio_plt["event_id"].value_counts().index[0]
piv = (account_plt[account_plt.event_id == ev]
       .pivot_table(index=["year_id", "loss_date"], columns="accnt_no", values="loss"))
print(f"within-event correlation of account losses, event {ev}:")
display(piv.corr().round(3))
print("\nIf these are ~0 the common shock is not wired through: the portfolio tail will be far")
print("too thin and every diversification and marginal-impact number downstream will be wrong.")

## Next step

`account_plt` and `portfolio_plt` are the two inputs the compression notebook expects — drop them
straight in and everything downstream (retention, factors, AEP sparse product, OEP blocked scan,
marginal impact at RP 200) runs unchanged.

**One thing to check before scaling up:** `elt.groupby("event_id").size().describe()`. The engine
loops events in Python and vectorises across occurrences within each, so a catalogue of tens of
thousands of events with narrow footprints runs slower per row than a smaller catalogue with wide
ones. If that bites, the event loop is what to parallelise — not the account loop inside it, which
must stay serial to keep the shared shock aligned.